In [1]:
#pip install xgboost

In [7]:
import os
import random
import pandas as pd
import numpy as np
from glob import glob
import time
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

In [3]:
random.seed(0)

# Most important features (top 5 by SHAP):
# doublecheck: Numerator of the MP2 t2-amplitude, two-electron integral <ik || ab>
# t2start: Initial MP2 t2-amplitude
# t2mag: Magnitude of the MP2 t2-amplitude
# orbdiff: Denominator of the MP2 t2-amplitude
# diag: Binary feature denoting whether a=b (virtual orbits are the same)
feat = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag']

# Feature order in X
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

# get training molecules
basis_sets = ['STO-3G', 'cc-pVDZ', 'aug-cc-pVDZ'] 

In [4]:
import os
print(os.getcwd())


/mnt/c/Users/Maxim/DDLUCJ/machine_learning


In [5]:
steps = [20, 40, 60, 80, 100]
for basis in basis_sets: 
    for n in steps:
        t1 = time.time()
        # get training molecules
        #fn = os.path.join(os.path.expanduser("~"), "DDLUCJ", "DDLUCJ_Models", f"faster_{basis}_data_splits_{n}.pkl")
        fn = f"/mnt/c/Users/Maxim/Documents/Toronto/Research/code/out/faster_{basis}_data_splits_{n}.pkl"
        with open(fn, "rb") as f:
            data = joblib.load(f)
        
        
        X_train = data["X_train"]
        y_train = data["y_train"]
        X_test = data["X_test"]
        y_test = data["y_test"]
    
         
        model = XGBRegressor(
            n_estimators=400,
            max_depth=12,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            reg_alpha=0.0,
            tree_method="hist",   # faster + smaller
            n_jobs=-1,
            random_state=42
        )
        
        model_pipeline = Pipeline([
            ('scaler', MinMaxScaler(feature_range=(-1,1))),
            ('regressor', model)
        ])
        
        # fit model
        model_pipeline.fit(X_train, y_train)
        y_pred = model_pipeline.predict(X_test)
        
        # compute performance metrics
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
    
        # save metrics
        print(f"N: {n}, MAE: {mae}, MSE: {mse}, R2: {r2}")
        with open(f"out/xgboost_{basis}_performance.txt", "a") as f:
            f.write(f"N: {n}, MAE: {mae}, MSE: {mse}, R2: {r2}\n")
        
        
        # save model and train test splits
        #joblib.dump(model_pipeline, f"out/xgboost_faster_sto_3g_pVDZ_model_{n}.pkl")
        
        #model.save_model(f"out/{basis}_xgb_model_{n}.json")
        t2 = time.time()
        print(f"{n} samples took {t2-t1} seconds\n")


N: 20, MAE: 0.00022612851535391197, MSE: 7.514063191818838e-07, R2: 0.9861144058567034
20 samples took 91.64164566993713 seconds

N: 40, MAE: 0.00020677704572376852, MSE: 5.722418814605142e-07, R2: 0.9894252705694457
40 samples took 7.059980154037476 seconds

N: 60, MAE: 0.00018529358581940235, MSE: 5.124560619471156e-07, R2: 0.9905300811148126
60 samples took 23.264155626296997 seconds

N: 80, MAE: 0.00018348025016214653, MSE: 4.777590185949703e-07, R2: 0.9911712642532307
80 samples took 28.422141313552856 seconds

N: 100, MAE: 0.0001749861279861395, MSE: 3.760582539233551e-07, R2: 0.9930506409715827
100 samples took 41.00954627990723 seconds

N: 20, MAE: 4.9548887360112236e-05, MSE: 1.578525723803782e-07, R2: 0.887979990222034
20 samples took 107.96486067771912 seconds



FileNotFoundError: [Errno 2] No such file or directory: '/mnt/c/Users/Maxim/Documents/Toronto/Research/code/out/faster_cc-pVDZ_data_splits_40.pkl'

In [ ]:
steps = [20, 40, 60, 80, 100]
for basis in basis_sets: 
    for n in steps:
        print(f"Basis: {basis}, N={n}")
        t1 = time.time()
        # get training molecules
        #fn = os.path.join(os.path.expanduser("~"), "DDLUCJ", "DDLUCJ_Models", f"faster_{basis}_data_splits_{n}.pkl")
        fn = f"/mnt/c/Users/Maxim/Documents/Toronto/Research/code/out/faster_{basis}_data_splits_{n}.pkl"
        with open(fn, "rb") as f:
            data = joblib.load(f)
        
        
        scaler = MinMaxScaler(feature_range=(-1,1))
        X_train = scaler.fit_transform(data["X_train"])
        y_train = data["y_train"]
        
        X_test = scaler.transform(data["X_test"])
        y_test = data["y_test"]
        
                
        params = {'max_depth': [1, 10, 100],
           'n_estimators': [100, 500, 1000],
               'reg_lambda': [1e-6, 1e-3,1e-1],
               'reg_alpha': [1e-6, 1e-3,1e-1]}
        
        model = XGBRegressor()
        grid = GridSearchCV(estimator=model, 
                        param_grid=params,
                        scoring='r2', 
                        verbose=1000,n_jobs=5).fit(X_train,y_train)
        
        
        model=grid.best_estimator_
        y_pred_train=model.predict(X_train)
        y_pred_test=model.predict(X_test)
        print(f"MAE: {mean_absolute_error(y_train, y_pred_train)}, {mean_absolute_error(y_test, y_pred_test)}")
        print(f"R2: {r2_score(y_train,y_pred_train):.4f},{r2_score(y_test,y_pred_test):.4f}")
        print(f"RMSE (mEh): {root_mean_squared_error(y_train,y_pred_train)*1e3:.4f},{root_mean_squared_error(y_test,y_pred_test)*1e3:.4f}")
        
        #model.save_model(f"out/{basis}_xgb_model_{n}.json")
        t2 = time.time()
        print(f"{n} samples took {t2-t1} seconds\n")

Basis: STO-3G, N=20
Fitting 5 folds for each of 81 candidates, totalling 405 fits
[CV 2/5; 40/81] START max_depth=10, n_estimators=500, reg_alpha=0.001, reg_lambda=1e-06
[CV 2/5; 40/81] END max_depth=10, n_estimators=500, reg_alpha=0.001, reg_lambda=1e-06;, score=0.996 total time=   0.7s
[CV 2/5; 41/81] START max_depth=10, n_estimators=500, reg_alpha=0.001, reg_lambda=0.001
[CV 2/5; 41/81] END max_depth=10, n_estimators=500, reg_alpha=0.001, reg_lambda=0.001;, score=0.996 total time=   0.6s
[CV 3/5; 42/81] START max_depth=10, n_estimators=500, reg_alpha=0.001, reg_lambda=0.1
[CV 3/5; 42/81] END max_depth=10, n_estimators=500, reg_alpha=0.001, reg_lambda=0.1;, score=0.998 total time=   0.6s
[CV 3/5; 43/81] START max_depth=10, n_estimators=500, reg_alpha=0.1, reg_lambda=1e-06
[CV 3/5; 43/81] END max_depth=10, n_estimators=500, reg_alpha=0.1, reg_lambda=1e-06;, score=0.984 total time=   0.6s
[CV 3/5; 44/81] START max_depth=10, n_estimators=500, reg_alpha=0.1, reg_lambda=0.001
[CV 3/5; 44/